# Tutorial: Exploring TCGA Data with the GDC Client

This notebook walks through how to query TCGA (The Cancer Genome Atlas) data using our GDC client.

**What you'll learn:**
- What data is available in TCGA
- How to discover fields dynamically (no memorizing!)
- How to query patients and their clinical data
- How to find slide images and other files

## 1. Setup

First, let's import the client and create a connection. No authentication needed for open-access data.

In [2]:
import sys
sys.path.insert(0, '../../../..')  # Add project root to path

from src.data.tcga import GDCClient

client = GDCClient()
print("Connected to GDC API")

Connected to GDC API


## 2. What Cancer Types Are Available?

TCGA has 33 cancer types. Let's see them all.

In [3]:
projects = client.list_projects(program="TCGA")

print(f"Found {len(projects)} TCGA projects\n")
print(f"{'Project':<12} | {'Patients':>8} | {'Files':>8} | Cancer Type")
print("-" * 70)

for p in projects:
    disease = p.disease_type[0] if p.disease_type else "N/A"
    print(f"{p.project_id:<12} | {p.case_count:>8} | {p.file_count:>8} | {disease[:35]}")

Found 33 TCGA projects

Project      | Patients |    Files | Cancer Type
----------------------------------------------------------------------
TCGA-LGG     |      516 |    33453 | Gliomas
TCGA-BRCA    |     1098 |    70774 | Adnexal and Skin Appendage Neoplasm
TCGA-LAML    |      200 |     8839 | Myeloid Leukemias
TCGA-UCS     |       57 |     3720 | Basal Cell Neoplasms
TCGA-GBM     |      617 |    30326 | Not Reported
TCGA-THYM    |      124 |     7968 | Thymic Epithelial Neoplasms
TCGA-TGCT    |      263 |    12851 | Germ Cell Neoplasms
TCGA-PCPG    |      179 |    11823 | Paragangliomas and Glomus Tumors
TCGA-CHOL    |       51 |     3171 | Adenomas and Adenocarcinomas
TCGA-DLBC    |       58 |     3141 | Mature B-Cell Lymphomas
TCGA-CESC    |      307 |    19315 | Squamous Cell Neoplasms
TCGA-ESCA    |      185 |    11120 | Squamous Cell Neoplasms
TCGA-ACC     |       92 |     5789 | Adenomas and Adenocarcinomas
TCGA-KICH    |      113 |     5993 | Adenomas and Adenocarcinomas
TC

## 3. Discovering Available Fields

You don't need to memorize field names. The client can tell you what's available.

**Key concept:** Fields can be "expanded" to include nested data (like patient demographics, diagnoses, samples).

In [4]:
# What nested data can we expand for cases (patients)?
expandable = client.get_expandable_fields("cases")

print("Expandable fields for CASES (patients):")
print("These are nested objects you can include in queries\n")

for field in expandable:
    print(f"  - {field}")

Expandable fields for CASES (patients):
These are nested objects you can include in queries

  - annotations
  - demographic
  - diagnoses
  - diagnoses.annotations
  - diagnoses.pathology_details
  - diagnoses.treatments
  - exposures
  - family_histories
  - files
  - files.analysis
  - files.analysis.input_files
  - files.analysis.metadata
  - files.analysis.metadata.read_groups
  - files.analysis.metadata.read_groups.read_group_qcs
  - files.archive
  - files.center
  - files.downstream_analyses
  - files.downstream_analyses.output_files
  - files.index_files
  - files.metadata_files
  - follow_ups
  - follow_ups.molecular_tests
  - follow_ups.other_clinical_attributes
  - project
  - project.program
  - samples
  - samples.annotations
  - samples.portions
  - samples.portions.analytes
  - samples.portions.analytes.aliquots
  - samples.portions.analytes.aliquots.annotations
  - samples.portions.analytes.aliquots.center
  - samples.portions.analytes.annotations
  - samples.portions.

In [ ]:
# What are ALL available fields? (there are hundreds)
all_fields = client.discover_fields("cases")

print(f"Total available fields: {len(all_fields)}\n")
print("First 30 fields:")
for f in all_fields[:30]:
    print(f"  {f}")
print(f"\n... and {len(all_fields) - 30} more")

## 4. Getting Patient Data

Let's get some actual patients from TCGA-BRCA (Breast Cancer).

We use `expand=` to tell the API what nested data to include.

In [ ]:
# Get 5 breast cancer patients with their clinical data
cases = client.get_cases(
    project_id="TCGA-BRCA",
    expand=["demographic", "diagnoses", "samples"],
    max_results=5
)

print(f"Retrieved {len(cases)} patients\n")

for c in cases:
    print(f"Patient: {c.submitter_id}")
    print(f"  Gender: {c.gender}")
    print(f"  Race: {c.race}")
    print(f"  Age at diagnosis: {c.age_at_diagnosis} days (~{c.age_at_diagnosis//365 if c.age_at_diagnosis else '?'} years)")
    print(f"  Cancer: {c.primary_diagnosis}")
    print(f"  Stage: {c.tumor_stage}")
    print(f"  Status: {c.vital_status}")
    print(f"  Samples collected: {len(c.samples)}")
    print(f"  Slides available: {len(c.slide_ids)}")
    print()

### Looking at Raw Data

Each case object has a `_raw` attribute with the complete API response. Useful for exploring what's really there.

In [ ]:
# Look at raw data for first patient
import json

case = cases[0]
print(f"Raw data keys for {case.submitter_id}:")
print(list(case._raw.keys()))

print("\nDemographic data:")
print(json.dumps(case._raw.get('demographic', {}), indent=2))

print("\nFirst diagnosis:")
if case._raw.get('diagnoses'):
    print(json.dumps(case._raw['diagnoses'][0], indent=2))

## 5. Filtering Patients

You can filter by various criteria like gender, vital status, etc.

In [ ]:
# Find deceased female patients
deceased = client.get_cases(
    project_id="TCGA-BRCA",
    gender="female",
    vital_status="Dead",
    expand=["demographic", "diagnoses"],
    max_results=10
)

print(f"Found {len(deceased)} deceased female patients\n")

for c in deceased:
    days = c.days_to_death or "unknown"
    years = f"({c.days_to_death // 365} years)" if c.days_to_death else ""
    print(f"{c.submitter_id}: died after {days} days {years}")

## 6. Finding Files (Slides, Clinical Data, etc.)

Let's see what file types are available and get some slide images.

In [ ]:
# What data types exist in TCGA-BRCA?
data_types = client.get_available_data_types(project_id="TCGA-BRCA")

print("Available data types in TCGA-BRCA:\n")
for dt in data_types:
    print(f"  - {dt}")

In [ ]:
# Get some slide images (open access only)
slides = client.get_slide_images(
    project_id="TCGA-BRCA",
    access="open",
    max_results=5
)

print(f"Found {len(slides)} slide images\n")

for s in slides:
    size_gb = s.file_size / 1e9
    print(f"File: {s.filename}")
    print(f"  Patient: {s.case_submitter_id}")
    print(f"  Type: {s.experimental_strategy}")
    print(f"  Size: {size_gb:.2f} GB")
    print(f"  Format: {s.data_format}")
    print(f"  Download URL: {client.get_download_url(s.file_id)}")
    print()

In [ ]:
# Get clinical supplement files (XMLs with detailed clinical info)
clinical_files = client.get_clinical_files(
    project_id="TCGA-BRCA",
    access="open",
    max_results=5
)

print(f"Found {len(clinical_files)} clinical files\n")

for f in clinical_files:
    print(f"{f.filename}")
    print(f"  Type: {f.data_type}")
    print(f"  Format: {f.data_format}")
    print()

## 7. Creating a Download Manifest

To download files in bulk, you create a manifest and use the `gdc-client` tool.

In [ ]:
# Get slides and create a manifest
slides = client.get_slide_images("TCGA-BRCA", access="open", max_results=3)

# Create manifest (just display it, don't save)
manifest = client.create_manifest(slides)

print("Download manifest:")
print("-" * 80)
print(manifest)
print("-" * 80)

print("\nTo download these files:")
print("1. Save manifest to file: manifest.txt")
print("2. Run: gdc-client download -m manifest.txt")

## 8. Summary

**Key methods:**

| Method | What it does |
|--------|-------------|
| `list_projects()` | List all cancer types |
| `discover_fields(endpoint)` | See all available fields |
| `get_expandable_fields(endpoint)` | See what nested data is available |
| `get_cases(project_id, expand=[...])` | Get patient clinical data |
| `get_slide_images(project_id)` | Get pathology slides |
| `get_clinical_files(project_id)` | Get clinical XMLs |
| `get_available_data_types(project_id)` | See what file types exist |
| `create_manifest(files)` | Create download manifest |

**Next steps:**
- Run the test suite: `python src/data/tcga/gdc_client.py`
- See the README for more details: `src/data/tcga/README.md`